In [1]:
# Install packages not pre-installed on Kaggle
%pip install -q vietocr jiwer


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.9/133.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 93.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import time
import random
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jiwer

# ── PIL._util compatibility patch ────────────────────────────────────────────
try:
    from PIL._util import is_directory as _test_pil
except ImportError:
    import os as _os
    from PIL import _util as _pil_util
    if not hasattr(_pil_util, 'is_path'):
        _pil_util.is_path = lambda f: isinstance(f, (bytes, str, _os.PathLike))
    if not hasattr(_pil_util, 'is_directory'):
        _pil_util.is_directory = lambda f: (isinstance(f, (bytes, str, _os.PathLike)) and _os.path.isdir(f))
    print('Applied PIL._util compatibility patch.')

from PIL import Image
import torch

# ── NumPy 2.0 compatibility patches ─────────────────────────────────────────
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'float': [np.float16, np.float32, np.float64],
        'complex': [np.complex64, np.complex128],
        'others': [np.bool_, np.bytes_, np.str_, np.object_, np.void, np.datetime64, np.timedelta64],
    }
if not hasattr(np, 'bool'): np.bool = bool
if not hasattr(np, 'int'): np.int = int
if not hasattr(np, 'float'): np.float = float

from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor
from vietocr.model.trainer import Trainer

warnings.filterwarnings('ignore')
print(f'Torch version: {torch.__version__}')
print('All imports OK.')


Applied PIL._util compatibility patch.
Torch version: 2.10.0+cu128
All imports OK.


In [3]:
seed = 59
KAGGLE_ROOT = '/kaggle/input/datasets/ghtgfuiss123/vn-handwritten-ocr-dataset'

CONFIG = {
    # ── Paths ─────────────────────────────────────────────────────────────────
    'train_path'    : f'{KAGGLE_ROOT}/data/data/processed/train_line.txt',
    'val_path'      : f'{KAGGLE_ROOT}/data/data/processed/val.txt',
    'test_path'     : f'{KAGGLE_ROOT}/data/data/processed/test.txt',
    'metadata_path' : f'{KAGGLE_ROOT}/data/data/processed/split_metadata.json',
    'checkpoint_dir': '/kaggle/working/models/baseline/',

    # ── Data ──────────────────────────────────────────────────────────────────
    'image_root'    : f'{KAGGLE_ROOT}/Dataset/Dataset/data',

    'image_height'   : 32,
    'image_max_width': 690,
    'image_min_width': 32,
    'max_label_len'  : 180,
    'grayscale'      : True,

    # ── Model ─────────────────────────────────────────────────────────────────
    'backbone'  : 'vgg19_bn',
    'pretrained': True,

    # ── Training ──────────────────────────────────────────────────────────────
    'batch_size' : 32,
    'max_lr'     : 3e-4,
    'total_iters': 50000,
    'valid_every': 500,
    'log_every'  : 200,
    'device'     : 'cuda:0',
}

print('CONFIG loaded.')
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)


CONFIG loaded.


## 1. Load Processed Data

In [4]:
def read_annotation_file(filepath):
    """Read a tab-separated annotation file: <image_path>\\t<transcription>"""
    entries = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.rstrip("\n")
            if not line.strip():
                continue
            parts = line.split("\t", maxsplit=1)
            if len(parts) != 2:
                print(f"WARNING: Skipping malformed line {line_num} in {filepath}")
                continue
            entries.append((parts[0].strip(), parts[1].strip()))
    return entries

In [5]:
def filter_by_label_len(entries, max_len):
    """Filter out entries where len(transcription) > max_len."""
    kept = [(img, txt) for img, txt in entries if len(txt) <= max_len]
    return kept, len(entries) - len(kept)

In [6]:
# ---- Load annotation files ----
train_data = read_annotation_file(CONFIG["train_path"])
val_data = read_annotation_file(CONFIG["val_path"])
test_data = read_annotation_file(CONFIG["test_path"])

print(f"Training text-lines : {len(train_data):,}")
print(f"Val text-lines       : {len(val_data):,}")
print(f"Test samples  : {len(test_data):,}")

# ---- max_label_len guard ----
max_len = CONFIG["max_label_len"]
train_data, train_dropped = filter_by_label_len(train_data, max_len)
val_data, val_dropped = filter_by_label_len(val_data, max_len)
test_data, test_dropped = filter_by_label_len(test_data, max_len)


print(f"\nmax_label_len guard ({max_len}):")
print(f"  Train dropped : {train_dropped:,}")
print(f"  Val dropped   : {val_dropped:,}")
print(f"  Test dropped  : {test_dropped:,}")
print(f"\nSau khi lọc:")
print(f"  Train samples : {len(train_data):,}")
print(f"  Val samples   : {len(val_data):,}")
print(f"  Test samples  : {len(test_data):,}")

print("\nSample annotations (first 3):")
for img, txt in train_data[:3]:
    print(f"  {img}  ->  {txt[:80]}{'...' if len(txt) > 80 else ''}")

Training text-lines : 13,090
Val text-lines       : 1,636
Test samples  : 1,637

max_label_len guard (180):
  Train dropped : 0
  Val dropped   : 0
  Test dropped  : 0

Sau khi lọc:
  Train samples : 13,090
  Val samples   : 1,636
  Test samples  : 1,637

Sample annotations (first 3):
  data/UIT_HWDB_line_flat__40_4.jpg  ->  chưa di dời. Như vậy, trước thời điểm cấm xe tải nặng lưu thông ở nội ô vào ban
  data/VNOnDB_line__20160604_0199_25463_tg_0_3.png  ->  chiều 15 - 3 - 2004. Theo thông tin ban đầu, phía nạn nhân khai nhận với một số ...
  data/VNOnDB_line__20151111_0050_25439_2_tg_4_0.png  ->  Đề nghị anh Đại liên hệ với chúng tôi : Công ty Xây lắp & vật tư xây


## 2. Prepare Annotation Files


In [7]:
def write_annotation_file(entries, image_root, output_path):
    """Write entries to VietOCR annotation file with absolute image paths."""
    abs_root = os.path.abspath(image_root)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for img_path, transcript in entries:
            if img_path.startswith('data/'):
                basename = img_path[len('data/'):]
                abs_img = os.path.join(abs_root, basename)
            else:
                abs_img = os.path.join(abs_root, img_path)
            f.write(f"{abs_img}\t{transcript}\n")
    return output_path


In [8]:
# On Kaggle, input is read-only. We save annotations to /kaggle/working/annotations/
working_annot_dir = '/kaggle/working/annotations/'
os.makedirs(working_annot_dir, exist_ok=True)

train_annotation_path = os.path.join(working_annot_dir, 'train_annotation.txt')
val_annotation_path = os.path.join(working_annot_dir, 'val_annotation.txt')

write_annotation_file(train_data, CONFIG['image_root'], train_annotation_path)
write_annotation_file(val_data, CONFIG['image_root'], val_annotation_path)

print(f'Training annotation  : {train_annotation_path}')
print(f'Validation annotation: {val_annotation_path}')

Training annotation  : /kaggle/working/annotations/train_annotation.txt
Validation annotation: /kaggle/working/annotations/val_annotation.txt


## 3. Build VietOCR Config

In [9]:
config = Cfg.load_config_from_name("vgg_seq2seq")

In [10]:
config['trainer']

{'batch_size': 32,
 'print_every': 200,
 'valid_every': 4000,
 'iters': 100000,
 'export': './weights/transformerocr.pth',
 'checkpoint': './checkpoint/transformerocr_checkpoint.pth',
 'log': './train.log',
 'metrics': None}

In [11]:
# Dataset
config["dataset"]["data_root"] = ""  # Empty for absolute paths
config["dataset"]["name"] = "vn_handwritten_ocr"
config["dataset"]["train_annotation"] = os.path.abspath(train_annotation_path)
config["dataset"]["valid_annotation"] = os.path.abspath(val_annotation_path)
config["dataset"]["image_height"] = CONFIG["image_height"]
config["dataset"]["image_max_width"] = CONFIG["image_max_width"]
config["dataset"]["image_min_width"] = CONFIG["image_min_width"]

# Training hyperparameters
config["trainer"]["batch_size"] = CONFIG["batch_size"]
config["trainer"]["iters"] = CONFIG["total_iters"]
config["trainer"]["valid_every"] = CONFIG["valid_every"]
config["trainer"]["print_every"] = CONFIG["log_every"]
config["optimizer"]["max_lr"] = CONFIG["max_lr"]

# Checkpoint / logging
checkpoint_dir_abs = os.path.abspath(CONFIG["checkpoint_dir"])
os.makedirs(checkpoint_dir_abs, exist_ok=True)

config["trainer"]["export"] = os.path.join(checkpoint_dir_abs, "best_model.pth")
config["trainer"]["log"] = os.path.join(checkpoint_dir_abs, "training_log")
config["trainer"]["checkpoint"] = os.path.join(checkpoint_dir_abs, "last_checkpoint.pth")
# Device
config["device"] = CONFIG["device"]

# Print summary
print("=" * 60)
print("VietOCR Configuration Summary")
print("=" * 60)
for section_name in ["dataset", "transformer", "optimizer", "trainer"]:
    if section_name in config:
        print(f"\n[{section_name}]")
        section = config[section_name]
        if isinstance(section, dict):
            for k, v in section.items():
                print(f"  {k}: {v}")
        else:
            print(f"  {section}")
print(f"\nbackbone: {config.get('backbone', 'N/A')}")
print(f"device  : {config.get('device', 'N/A')}")
print("=" * 60)

VietOCR Configuration Summary

[dataset]
  name: vn_handwritten_ocr
  data_root: 
  train_annotation: /kaggle/working/annotations/train_annotation.txt
  valid_annotation: /kaggle/working/annotations/val_annotation.txt
  image_height: 32
  image_min_width: 32
  image_max_width: 690

[transformer]
  encoder_hidden: 256
  decoder_hidden: 256
  img_channel: 256
  decoder_embedded: 256
  dropout: 0.1

[optimizer]
  max_lr: 0.0003
  pct_start: 0.1

[trainer]
  batch_size: 32
  print_every: 200
  valid_every: 500
  iters: 50000
  export: /kaggle/working/models/baseline/best_model.pth
  checkpoint: /kaggle/working/models/baseline/last_checkpoint.pth
  log: /kaggle/working/models/baseline/training_log
  metrics: None

backbone: vgg19_bn
device  : cuda:0


## 4. Initialize Trainer

In [12]:
# Removed: Unnecessary train.txt overwrite
# Data already properly formatted in train_annotation.txt

In [13]:
import glob
import shutil
import os

def cleanup_lmdb_cache(patterns):
    removed = []
    for pattern in patterns:
        for path in glob.glob(pattern):
            if os.path.isdir(path):
                shutil.rmtree(path, ignore_errors=True)
                removed.append(path)
            elif os.path.isfile(path):
                os.remove(path)
                removed.append(path)
    return removed

# Remove stale VietOCR LMDB caches before Trainer() creates them.
# Corrupted LMDB usually appears after an interrupted previous run.
lmdb_patterns = [
    "train_data*",
    "valid_data*",
    "train_*",
    "valid_*",
    "/content/train_data*",
    "/content/valid_data*",
    "/content/train_*",
    "/content/valid_*",
    "/tmp/train_data*",
    "/tmp/valid_data*",
    "/tmp/train_*",
    "/tmp/valid_*",
]

removed = cleanup_lmdb_cache(lmdb_patterns)
if removed:
    print("Removed stale LMDB cache paths:")
    for path in removed:
        print(f"  - {path}")
else:
    print("No stale LMDB cache paths found.")

# Use a unique cache name to avoid colliding with previous interrupted runs.
config["dataset"]["name"] = f"vn_handwritten_ocr_seed_{seed}"
print(f'Dataset LMDB cache name: {config["dataset"]["name"]}')


No stale LMDB cache paths found.
Dataset LMDB cache name: vn_handwritten_ocr_seed_59


In [14]:
# Force cleanup old LMDB caches (may have the cnt-1 bug)
print("🗑️  Cleaning up old LMDB caches...")
import glob
import shutil

lmdb_patterns = [
    "train_vn_handwritten_ocr*",
    "valid_vn_handwritten_ocr*",
    "train_data*",
    "valid_data*"
]

removed_count = 0
for pattern in lmdb_patterns:
    for path in glob.glob(pattern):
        if os.path.isdir(path):
            print(f"   Removing: {path}")
            shutil.rmtree(path, ignore_errors=True)
            removed_count += 1

if removed_count > 0:
    print(f"✓ Removed {removed_count} old LMDB cache(s)")
else:
    print("✓ No old caches found")

print()

class Custom_Trainer(Trainer):
    def __init__(self, config, pretrained=True):
        super().__init__(config, pretrained)

    def train(self):
        total_loss = 0
        total_loader_time = 0
        total_gpu_time = 0
        best_acc = 0

        data_iter = iter(self.train_gen)

        try:
            for i in range(self.num_iters):
                self.iter += 1

                start = time.time()

                try:
                    batch = next(data_iter)
                except StopIteration:
                    data_iter = iter(self.train_gen)
                    try:
                        batch = next(data_iter)
                    except StopIteration:
                        print(f"\nDataset exhausted at iteration {self.iter}/{self.num_iters}")
                        break

                total_loader_time += time.time() - start

                start = time.time()
                loss = self.step(batch)
                total_gpu_time += time.time() - start

                total_loss += loss
                self.train_losses.append((self.iter, loss))

                if self.iter % self.print_every == 0:
                    info = (
                        "iter: {:06d} - train loss: {:.3f} - lr: {:.2e} - load time: {:.2f} - gpu time: {:.2f}".format(
                            self.iter,
                            total_loss / self.print_every,
                            self.optimizer.param_groups[0]["lr"],
                            total_loader_time,
                            total_gpu_time,
                        )
                    )

                    total_loss = 0
                    total_loader_time = 0
                    total_gpu_time = 0

                    print(info)
                    self.logger.log(info)

                    # SAVE CHECKPOINT
                    if self.checkpoint:
                        print(">>> Auto-saving checkpoint...")
                        self.save_checkpoint(self.checkpoint)

                if self.valid_annotation and self.iter % self.valid_every == 0:
                    val_loss = self.validate()
                    acc_full_seq, acc_per_char = self.precision(self.metrics)

                    info = (
                        "iter: {:06d} - valid loss: {:.3f} - acc full seq: {:.4f} - acc per char: {:.4f}".format(
                            self.iter, val_loss, acc_full_seq, acc_per_char
                        )
                    )
                    print(info)
                    self.logger.log(info)

                    if acc_full_seq > best_acc:
                        self.save_weights(self.export_weights)
                        best_acc = acc_full_seq

        finally:
            # 🔥 SAVE khi bị kill
            if self.checkpoint:
                try:
                    print(">>> Saving final checkpoint...")
                    self.save_checkpoint(self.checkpoint)
                except Exception as e:
                    print("Failed to save final checkpoint:", e)

🗑️  Cleaning up old LMDB caches...
✓ No old caches found



In [15]:
import shutil
import glob
import time

# ── Strong LMDB Cleanup ──────────────────────────────────────────────────
print('🧹  Strong LMDB Cache Cleanup...')
for d in glob.glob('train_*') + glob.glob('valid_*'):
    if os.path.exists(d):
        print(f'   Removing {d}')
        shutil.rmtree(d, ignore_errors=True)

# Use a timestamp-based unique name to avoid cache collision
config['dataset']['name'] = f'baseline_ocr_{int(time.time())}'
print(f'Using unique dataset name: {config["dataset"]["name"]}')

trainer = Custom_Trainer(config, pretrained=CONFIG["pretrained"])

# Validate dataset is loaded
try:
    dataset_size = len(trainer.train_gen.dataset)
    print(f"✓ Training dataset loaded: {dataset_size:,} samples")
    if dataset_size == 0:
        raise ValueError("Training dataset is empty! Check data paths and LMDB cache.")
except Exception as e:
    print(f"✗ Error checking dataset: {e}")
    raise

total_params = sum(p.numel() for p in trainer.model.parameters())
trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Frozen parameters    : {total_params - trainable_params:,}")
print(f"\nModel exported to    : {config['trainer']['export']}")
print(f"Training log at      : {config['trainer']['log']}")
print(f"Last checkpoint at   : {config['trainer']['checkpoint']}")

🧹  Strong LMDB Cache Cleanup...
Using unique dataset name: baseline_ocr_1776668990
Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth


100%|██████████| 548M/548M [00:03<00:00, 177MB/s]
10935it [00:11, 958.26it/s] 
Create train_baseline_ocr_1776668990: 100%|███████████████| 13090/13090 [00:00<00:00, 163611.99it/s]

Remove 13090 invalid images
Created dataset with -1 samples



train_baseline_ocr_1776668990 build cluster: 0it [00:00, ?it/s]
Create valid_baseline_ocr_1776668990: 100%|█████████████████████| 1636/1636 [00:29<00:00, 55.62it/s]


Created dataset with 1635 samples


valid_baseline_ocr_1776668990 build cluster: 100%|██████████| 1635/1635 [00:00<00:00, 193109.01it/s]

✗ Error checking dataset: __len__() should return >= 0


ValueError: __len__() should return >= 0

## 5. Training

In [ ]:
print("Starting training...")
print(f"  Total iterations : {CONFIG['total_iters']:,}")
print(f"  Batch size       : {CONFIG['batch_size']}")
print(f"  Validate every   : {CONFIG['valid_every']:,} iters")
print(f"  Log every        : {CONFIG['log_every']} iters")
print(f"  Device           : {CONFIG['device']}")
print("=" * 60)

try:
    trainer.train()
    print("\nTraining completed successfully!")
except KeyboardInterrupt:
    print("\nTraining interrupted by user. Last checkpoint saved.")
    trainer.save_checkpoint(config["trainer"]["checkpoint"])
except Exception as e:
    print(f"\nTraining failed with error: {e}")
    raise

## 6. Post-Training: Visualize Metrics

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt

log_dir = config["trainer"]["log"]
metrics_records = []

log_file_candidates = [
    log_dir,
    log_dir + ".log",
    os.path.join(log_dir, "train.log"),
    os.path.join(checkpoint_dir_abs, "train.log"),
]

log_content = None
for candidate in log_file_candidates:
    if os.path.isfile(candidate):
        with open(candidate, "r", encoding="utf-8") as f:
            log_content = f.read()
        print(f"Found training log at: {candidate}")
        break


# =====================================================
# 1️⃣ PARSE TRAIN LOSS FROM LOG
# =====================================================
if log_content:
    train_pattern = re.compile(
        r"iter:\s*(\d+)\s*-\s*train loss:\s*([\d.]+)",
        re.IGNORECASE
    )

    matches = train_pattern.findall(log_content)

    for iter_str, loss_str in matches:
        metrics_records.append({
            "iteration": int(iter_str),
            "loss": float(loss_str)
        })


# =====================================================
# 2️⃣ FALLBACK: trainer.train_losses
#    DẠNG: [(iter, loss), (iter, loss), ...]
# =====================================================
if not metrics_records and hasattr(trainer, "train_losses") and trainer.train_losses:

    print("Using trainer.train_losses as fallback")

    for item in trainer.train_losses:

        # Nếu là tuple (iteration, loss)
        if isinstance(item, (list, tuple)) and len(item) == 2:
            iteration, loss = item
        else:
            continue

        try:
            metrics_records.append({
                "iteration": int(iteration),
                "loss": float(loss)
            })
        except:
            continue


# =====================================================
# 3️⃣ CREATE DATAFRAME
# =====================================================
if metrics_records:

    df_metrics = (
        pd.DataFrame(metrics_records)
        .drop_duplicates(subset="iteration")
        .sort_values("iteration")
        .reset_index(drop=True)
    )

    df_metrics["iteration"] = pd.to_numeric(df_metrics["iteration"], errors="coerce")
    df_metrics["loss"] = pd.to_numeric(df_metrics["loss"], errors="coerce")

    df_metrics = df_metrics.dropna()

    # =====================================================
    # 4️⃣ PLOT
    # =====================================================
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(
        df_metrics["iteration"],
        df_metrics["loss"],
        linewidth=1,
        alpha=0.7,
        label="Train Loss"
    )

    window = max(1, len(df_metrics) // 20)

    if len(df_metrics) > window:
        smoothed = df_metrics["loss"].rolling(
            window=window,
            min_periods=1
        ).mean()

        ax.plot(
            df_metrics["iteration"],
            smoothed,
            linewidth=2,
            color="red",
            label=f"Smoothed (w={window})"
        )

    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss Curve")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(checkpoint_dir_abs, "loss_curve.png"), dpi=150)
    plt.show()

    # =====================================================
    # 5️⃣ SAVE CSV
    # =====================================================
    metrics_csv_path = os.path.join(checkpoint_dir_abs, "metrics.csv")
    df_metrics.to_csv(metrics_csv_path, index=False)

    print(f"Metrics saved to: {metrics_csv_path}")
    print("\nMetrics summary:")
    print(df_metrics.describe())

else:
    print("No training metrics found.")
    df_metrics = pd.DataFrame(columns=["iteration", "loss"])
    metrics_csv_path = os.path.join(checkpoint_dir_abs, "metrics.csv")
    df_metrics.to_csv(metrics_csv_path, index=False)
    print(f"Empty metrics CSV saved to: {metrics_csv_path}")

In [ ]:
def _resolve_image_path(image_path: str, image_root: str) -> str:
    """Ghép đường dẫn ảnh an toàn, tránh lặp data/data."""
    rel_path = image_path
    if rel_path.startswith("data/"):
        rel_path = rel_path[len("data/"):]
    return os.path.join(os.path.abspath(image_root), rel_path)

In [ ]:
def compute_cer(predictions: list, ground_truths: list) -> float:
    """Compute Character Error Rate."""
    assert len(predictions) == len(ground_truths)
    filtered_preds, filtered_gts = [], []
    for pred, gt in zip(predictions, ground_truths):
        gt_clean = gt.strip()
        pred_clean = pred.strip()
        if len(gt_clean) == 0:
            continue
        filtered_preds.append(pred_clean)
        filtered_gts.append(gt_clean)
    if len(filtered_gts) == 0:
        return 0.0
    return float(jiwer.cer(filtered_gts, filtered_preds))

In [ ]:
def compute_wer(predictions: list, ground_truths: list) -> float:
    """Compute WER (error rate at token level)."""
    assert len(predictions) == len(ground_truths)
    filtered_preds, filtered_gts = [], []
    for pred, gt in zip(predictions, ground_truths):
        gt_clean = gt.strip()
        pred_clean = pred.strip()
        if len(gt_clean) == 0 or len(gt_clean.split()) == 0:
            continue
        filtered_preds.append(pred_clean)
        filtered_gts.append(gt_clean)
    if len(filtered_gts) == 0:
        return 0.0
    return float(jiwer.wer(filtered_gts, filtered_preds))

In [ ]:
best_model_path = config["trainer"]["export"]

if os.path.isfile(best_model_path):
    print(f"Loading best model from: {best_model_path}")

    pred_config = Cfg.load_config_from_name("vgg_seq2seq")
    pred_config["dataset"]["image_height"] = CONFIG["image_height"]
    pred_config["dataset"]["image_max_width"] = CONFIG["image_max_width"]
    pred_config["dataset"]["image_min_width"] = CONFIG["image_min_width"]
    pred_config["weights"] = best_model_path
    pred_config["device"] = CONFIG["device"]

    predictor = Predictor(pred_config)
    print("Predictor initialized.")

    predictions = []
    ground_truths = []
    image_paths = []
    errors=[]

    test_image_root = os.path.abspath(CONFIG["image_root"])
    use_grayscale = CONFIG["grayscale"]

    for img_rel_path, gt_text in test_data:
        img_full_path = _resolve_image_path(img_rel_path, CONFIG["image_root"])

        try:
            img = Image.open(img_full_path)
            if use_grayscale:
                img = img.convert("L").convert("RGB")
            else:
                img = img.convert("RGB")
            pred_text = predictor.predict(img)
            predictions.append(pred_text)
            ground_truths.append(gt_text)
            image_paths.append(img_rel_path)
        except FileNotFoundError:
            errors.append((img_rel_path, "FileNotFoundError"))
        except Exception as e:
            errors.append((img_rel_path, str(e)))

    if predictions:
        test_cer = compute_cer(predictions, ground_truths)
        test_wer = compute_wer(predictions, ground_truths)

        df_results = pd.DataFrame({
            "image": image_paths,
            "ground_truth": ground_truths,
            "prediction": predictions,
            "match": [p.strip() == g.strip() for p, g in zip(predictions, ground_truths)],
        })
        df_results["exact_match"] = df_results["prediction"].str.strip() == df_results["ground_truth"].str.strip()

        exact_match = int(df_results["exact_match"].sum())
        exact_match_pct = exact_match / len(df_results) * 100

        print(f"\nTest Evaluation ({len(df_results):,} samples):")
        print(f"  CER: {test_cer:.4f} ({test_cer * 100:.2f}%)")
        print(f"  WER: {test_wer:.4f} ({test_wer * 100:.2f}%)")
        print(f"  Exact matches: {exact_match}/{len(df_results)} ({exact_match_pct:.2f}%)")
        print(f"  Prediction errors: {len(errors):,}")
        if use_grayscale:
            print("  Grayscale: enabled (L -> RGB)")

        test_eval_csv = os.path.join(checkpoint_dir_abs, "test_predictions.csv")
        df_results.to_csv(test_eval_csv, index=False, encoding="utf-8")
        print(f"\nĐã lưu dự đoán test tại: {test_eval_csv}")

        if errors:
            print("\n5 lỗi đầu tiên:")
            for p, e in errors[:5]:
                print(f"  {p}: {e}")
    else:
        print("Không tạo được dự đoán hợp lệ trên test set. Hãy kiểm tra đường dẫn ảnh.")
else:
    print(f"Không tìm thấy best model tại: {best_model_path}")
    print("Bỏ qua test evaluation. Hãy chạy lại cell này sau khi train xong.")